# Adrianic v4.2 — Direct Semantic / Compositional Probe

This version asks exact, rephrased, two-hop, inverse, and negative-control questions. Probe-chain composed/inverse target sentences are omitted from training. No semantic labels or answer keys enter training or memory selection.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter
import argparse, json, math, random, re, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

if Path('/content').exists(): ROOT=Path('/content')
elif Path('/mnt/data').exists(): ROOT=Path('/mnt/data')
else: ROOT=Path.cwd()
OUT=ROOT/'adrianic_v42_semantic_probe_results'; OUT.mkdir(parents=True,exist_ok=True)
TOKEN_RE=re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP=set('the a an is are in of to and this that with for near under before after through beyond'.split())

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
def tokenize(s): return TOKEN_RE.findall(s.lower())

FACTS=[
('two plus two equals','four'),('three plus five equals','eight'),('seven minus three equals','four'),('six times seven equals','fortytwo'),
('the square root of nine is','three'),('a triangle has','three'),('a square has','four'),('the additive identity is','zero'),
('the si unit of force is the','newton'),('the si unit of energy is the','joule'),('the si unit of power is the','watt'),('mass times acceleration gives','force'),
('energy associated with motion is','kinetic'),('the force attracting masses is','gravity'),('the chemical symbol for hydrogen is','h'),
('the chemical symbol for oxygen is','o'),('water contains hydrogen and','oxygen'),('an atom with six protons is','carbon'),
('an atom with eight protons is','oxygen'),('the center of an atom is the','nucleus'),('earth orbits the','sun'),('the moon orbits the','earth'),
('the largest planet in the solar system is','jupiter'),('plants use light in','photosynthesis')]
REPHRASE=[
('in mechanics mass times acceleration produces','force'),('force is measured using the unit called','newton'),
('energy uses the si unit called','joule'),('water contains hydrogen plus','oxygen'),('six protons identifies','carbon'),
('eight protons identifies','oxygen'),('earth travels around the','sun'),('the moon travels around the','earth')]

SYL1=['nar','vel','tor','lum','kas','mer','sol','dav','pir','zan','fel','rin','bos','kel','vim','jor','tav','qul','ser','pan','dru','hel','cor','nis']
SYL2=['a','en','or','is','um','et','on','ar','il','os','ex','un']
def unique_words(n,seed):
    rng=random.Random(seed); pool=[a+b for a in SYL1 for b in SYL2]; rng.shuffle(pool); return pool[:n]
def make_chains(seed=0,n_demo=20,n_probe=10):
    names=unique_words((n_demo+n_probe)*3,seed+818); demo=[]; probe=[]; rows=[]; k=0
    for i in range(n_demo+n_probe):
        x,y,z=names[k:k+3]; k+=3
        premises=[f'{x} contains {y}.',f'{y} maps to {z}.']
        if i<n_demo:
            demo += (premises+[f'{x} ultimately maps to {z}.',f'{y} is contained by {x}.'])*3
            rows.append(('demo',x,y,z))
        else:
            probe += premises*4; rows.append(('probe',x,y,z))
    return demo,probe,pd.DataFrame(rows,columns=['kind','x','y','z'])
def make_training_corpus(seed=0):
    rng=random.Random(seed); s=[]
    for p,a in FACTS: s += [f'{p} {a}.',f'fact: {p} {a}.',f'remember {p} {a}.']
    demo,probe,chains=make_chains(seed); s += demo+probe
    for i in range(80): s.append(f'noise{i} drift{i} hollow{i} amber{i} sevenfold{i}.')
    rng.shuffle(s); return s,chains

def proxy_stream(seed=0,n=240):
    rng=random.Random(seed+500)
    names=['arden','cassian','lyra','orion','mara','selene','tavian','viola']; verbs=['whispers','wanders','argues','laughs','waits','returns','dreams','speaks']
    objs=['beside the gate','through the hall','near the river','under the tower','after the feast','beyond the garden','before the crowd','under the moon']
    return [f'{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}.' for _ in range(n)]
def download_natural():
    urls={'shakespeare':'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt','bible':'https://www.gutenberg.org/cache/epub/10900/pg10900.txt'}; out={}
    for name,url in urls.items():
        p=ROOT/f'{name}_corpus.txt'
        if not p.exists(): urllib.request.urlretrieve(url,p)
        text=p.read_text(errors='ignore'); out[name]=[q.strip().replace('\n',' ') for q in re.split(r'(?<=[.!?])\s+|\n{2,}',text) if 4<=len(tokenize(q))<=70]
    return out
def natural_stream(seed=0,corpus='interleaved',per_source=700):
    d=download_natural(); rng=random.Random(seed+900)
    A=rng.sample(d['shakespeare'],min(per_source,len(d['shakespeare']))); B=rng.sample(d['bible'],min(per_source,len(d['bible'])))
    if corpus=='shakespeare': return A
    if corpus=='bible': return B
    out=[]; n=min(len(A),len(B))
    for a,b in zip(A[:n],B[:n]): out += ([a,b] if rng.random()<.5 else [b,a])
    return out

class Vocab:
    def __init__(self,texts,max_vocab=9000):
        c=Counter(); [c.update(tokenize(t)) for t in texts]; self.itos=['<pad>','<unk>']+[w for w,_ in c.most_common(max_vocab-2)]; self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,s): return [self.stoi.get(w,1) for w in tokenize(s)]
class TinySSM(nn.Module):
    def __init__(self,V,emb=36,hid=56):
        super().__init__(); self.emb=nn.Embedding(V,emb); self.rnn=nn.RNN(emb,hid,nonlinearity='tanh',batch_first=True); self.out=nn.Linear(hid,V)
    def forward(self,x,h=None): z,h=self.rnn(self.emb(x),h); return self.out(z),h
def dev(model): return next(model.parameters()).device
def train_sentence(model,opt,ids):
    if len(ids)<2:return
    d=dev(model); x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:]; y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    opt.zero_grad(set_to_none=True); logits,_=model(x); loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
def seq_loss(model,ids):
    if len(ids)<2:return 0.0
    d=dev(model); x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:]; y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad(): logits,_=model(x)
    return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)))
def token_rank(model,vocab,prompt,answer):
    ids=vocab.encode(prompt); d=dev(model); x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad(): logits,_=model(x)
    v=logits[0,-1]; aid=vocab.stoi.get(answer,1); return int((v>v[aid]).sum().item())+1
def top_tokens(model,vocab,prompt,k=5):
    ids=vocab.encode(prompt); d=dev(model); x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad(): logits,_=model(x)
    return [vocab.itos[i] for i in torch.topk(logits[0,-1],k=min(k,len(vocab.itos))).indices.tolist()]

def chunk_features(sentences,vocab,model):
    counts=Counter(); toks=[]
    for s in sentences:
        q=[w for w in tokenize(s) if w.isalpha() and w not in STOP]; toks.append(q); counts.update(set(q))
    rows=[]
    for i,(s,q) in enumerate(zip(sentences,toks)):
        rows.append({'idx':i,'recurrence':np.mean([math.log1p(counts[w]) for w in q]) if q else 0.,'cross_context':np.mean([counts[w]>=3 for w in q]) if q else 0.,'predictability':-seq_loss(model,vocab.encode(s))})
    return pd.DataFrame(rows)
def evidence_score(df):
    X=df[['recurrence','cross_context','predictability']].to_numpy(float); mu=X.mean(0); sd=X.std(0); active=sd>1e-8; Z=np.zeros_like(X); Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)
    C=np.zeros((3,3)); idx=np.where(active)[0]
    if len(idx)>1:
        Q=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False))); np.fill_diagonal(Q,0.)
        for a,i in enumerate(idx):
            for b,j in enumerate(idx): C[i,j]=Q[a,b]
    rho=np.where(active,.7,0.); a=rho**3
    if a.sum()<1e-12:return np.zeros(len(df))
    base=a/(a.sum()+1e-12); w=a/(1+1.5*(C@base)); w/=w.sum()+1e-12; return Z@w
class Memory:
    PRIME=1000003
    def __init__(self): self.items=[]
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64); return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def add(self,ids,idx): p,q=self.chk(ids); self.items.append({'a':list(ids),'b':list(ids),'p':p,'q':q,'idx':idx})
    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for it in self.items:
            if it['a'] and rng.random()<p:
                j=rng.randrange(len(it['a'])); it['a'][j]=(it['a'][j]+rng.randint(1,19))%100000
    def get(self,parity=True):
        out=[]; repair=lost=0
        for it in self.items:
            if self.chk(it['a'])==(it['p'],it['q']): out.append(it['a'])
            elif parity and self.chk(it['b'])==(it['p'],it['q']): out.append(it['b']); repair+=1
            else: lost+=1
        return out,repair,lost
def build_memory(train,vocab,model,keep=.30):
    feat=chunk_features(train,vocab,model); score=evidence_score(feat); feat['score']=score; sets=[set(w for w in tokenize(s) if w.isalpha() and w not in STOP) for s in train]
    k=int(round(keep*len(train))); selected=[]; remain=set(range(len(train)))
    while remain and len(selected)<k:
        best=None; bv=-1e99
        for i in remain:
            red=max((len(sets[i]&sets[j])/(len(sets[i]|sets[j])+1e-12) for j in selected),default=0.); val=float(score[i])-.85*red
            if val>bv: bv=val; best=i
        selected.append(best); remain.remove(best)
    selected=set(selected); lo=np.quantile(score,.25); mem=Memory(); decision=[]
    for i,s in enumerate(train):
        if i in selected: decision.append('INCLUDE'); mem.add(vocab.encode(s),i)
        elif score[i]<=lo: decision.append('EXCLUDE')
        else: decision.append('VALUE_UNRESOLVED')
    feat['decision']=decision; return mem,feat

def probe_panel(model,vocab,chains,seed,stage,variant):
    rows=[]
    def add(kind,prompt,ans,item):
        r=token_rank(model,vocab,prompt,ans); rows.append({'seed':seed,'stage':stage,'variant':variant,'probe':kind,'item':item,'prompt':prompt,'answer':ans,'rank':r,'top1':int(r==1),'top5':int(r<=5),'top_tokens':'|'.join(top_tokens(model,vocab,prompt,5))})
    for i,(p,a) in enumerate(FACTS): add('exact_fact',p,a,f'fact{i}')
    for i,(p,a) in enumerate(REPHRASE): add('rephrase',p,a,f'rephrase{i}')
    q=chains[chains.kind=='probe'].reset_index(drop=True)
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        add('two_hop',f'{row.x} ultimately maps to',row.z,f'chain{i}')
        add('two_hop_shuffled',f'{row.x} ultimately maps to',wrong.z,f'chain{i}')
        add('inverse',f'{row.y} is contained by',row.x,f'chain{i}')
        add('inverse_shuffled',f'{row.y} is contained by',wrong.x,f'chain{i}')
        add('premise_contains',f'{row.x} contains',row.y,f'chain{i}'); add('premise_maps',f'{row.y} maps to',row.z,f'chain{i}')
    return pd.DataFrame(rows)
def summarize_probes(df):
    return df.groupby(['stage','variant','probe']).agg(top1=('top1','mean'),top5=('top5','mean'),mrr=('rank',lambda x:float(np.mean(1/np.asarray(x,float)))),median_rank=('rank','median'),n=('rank','size')).reset_index()

VARIANTS=['baseline','random_replay','framework','framework_no_parity']
def run_one(seed=0,full=False,corpus='interleaved',device=None):
    seed_all(seed); train,chains=make_training_corpus(seed); stream=natural_stream(seed,corpus,700) if full else proxy_stream(seed,240); vocab=Vocab(train+stream)
    if device is None: device='cuda' if full and torch.cuda.is_available() else 'cpu'
    runs=[]; probes=[]; audits=[]
    for variant in VARIANTS:
        seed_all(seed); model=TinySSM(len(vocab.itos)).to(device); opt=torch.optim.Adam(model.parameters(),lr=.006); order=list(range(len(train)))
        for ep in range(3):
            random.Random(seed+ep).shuffle(order)
            for i in order: train_sentence(model,opt,vocab.encode(train[i]))
        probes.append(probe_panel(model,vocab,chains,seed,'pre',variant)); mem,feat=build_memory(train,vocab,model,.30)
        if variant=='random_replay':
            n=len(mem.items); picks=random.Random(seed+999).sample(range(len(train)),n); mem=Memory()
            for i in picks: mem.add(vocab.encode(train[i]),i)
        if variant=='framework':
            aud=feat.copy(); aud['text']=[train[i] for i in aud.idx]; aud['seed']=seed; audits.append(aud)
        mem.corrupt(.18,seed+1234); replay,repair,lost=mem.get(parity=(variant!='framework_no_parity')); total=0
        for ep in range(2 if full else 1):
            for s in stream:
                train_sentence(model,opt,vocab.encode(s)); total+=1
                if variant!='baseline' and replay and total%6==0: train_sentence(model,opt,replay[(total//6)%len(replay)])
        post=probe_panel(model,vocab,chains,seed,'post',variant); probes.append(post)
        def metric(kind):
            q=post[post.probe==kind]; return float(np.mean(1/q['rank'].to_numpy(float)))
        runs.append({'seed':seed,'variant':variant,'full':full,'corpus':corpus,'exact_mrr':metric('exact_fact'),'rephrase_mrr':metric('rephrase'),'twohop_mrr':metric('two_hop'),'inverse_mrr':metric('inverse'),'premise_contains_mrr':metric('premise_contains'),'premise_maps_mrr':metric('premise_maps'),'twohop_top1':float(post[post.probe=='two_hop'].top1.mean()),'inverse_top1':float(post[post.probe=='inverse'].top1.mean()),'parity_repaired':repair,'parity_lost':lost,'stream_loss':float(np.mean([seq_loss(model,vocab.encode(s)) for s in stream[-60:]]))})
    return pd.DataFrame(runs),pd.concat(probes,ignore_index=True),pd.concat(audits,ignore_index=True)

def main(argv=None):
    ap=argparse.ArgumentParser(); ap.add_argument('--full',action='store_true'); ap.add_argument('--corpus',choices=['shakespeare','bible','interleaved'],default='interleaved'); ap.add_argument('--seeds',type=int,default=3); args=ap.parse_args(argv)
    rr=[]; pp=[]; aa=[]
    for seed in range(args.seeds):
        print('seed',seed); r,p,a=run_one(seed,args.full,args.corpus); rr.append(r); pp.append(p); aa.append(a); print(r[['variant','exact_mrr','rephrase_mrr','twohop_mrr','inverse_mrr']].to_string(index=False))
    runs=pd.concat(rr,ignore_index=True); probes=pd.concat(pp,ignore_index=True); audit=pd.concat(aa,ignore_index=True); tag=('full_'+args.corpus) if args.full else 'proxy'
    runs.to_csv(OUT/f'{tag}_{args.seeds}seed_runs.csv',index=False); probes.to_csv(OUT/f'{tag}_{args.seeds}seed_probe_details.csv',index=False); audit.to_csv(OUT/f'{tag}_{args.seeds}seed_memory_audit.csv',index=False); summarize_probes(probes).to_csv(OUT/f'{tag}_{args.seeds}seed_probe_summary.csv',index=False)
    med=runs.groupby('variant').median(numeric_only=True)
    checks={'framework_exact_retention_beats_baseline':bool(med.loc['framework','exact_mrr']>med.loc['baseline','exact_mrr']),'framework_rephrase_beats_baseline':bool(med.loc['framework','rephrase_mrr']>med.loc['baseline','rephrase_mrr']),'framework_twohop_beats_baseline':bool(med.loc['framework','twohop_mrr']>med.loc['baseline','twohop_mrr']),'framework_inverse_beats_baseline':bool(med.loc['framework','inverse_mrr']>med.loc['baseline','inverse_mrr']),'twohop_signal_nontrivial':bool(med.loc['framework','twohop_mrr']>.02),'parity_repairs_faults':bool(med.loc['framework','parity_repaired']>0)}
    (OUT/f'{tag}_{args.seeds}seed_checks.json').write_text(json.dumps(checks,indent=2))
    report=f'''Adrianic v4.2 — Direct Semantic / Compositional Probe\n\nMode: {tag}\nSeeds: {args.seeds}\n\nTraining is next-token prediction only. No semantic labels, chain IDs, relation roles, answer keys, or probe targets enter optimization or memory selection.\n\nProbe classes:\n- exact_fact: seen factual relationship\n- rephrase: unseen wording of an ordinary fact\n- two_hop: X contains Y; Y maps to Z; ask X ultimately maps to ?. Probe-chain X->Z sentence is omitted from training.\n- inverse: X contains Y; ask Y is contained by ?. Probe-chain inverse sentence is omitted from training.\n- premise controls: verify the two premises themselves survived.\n\n20 demonstration chains expose the generic linguistic schemas. Ten disjoint probe chains contain only the premises. Thus success tests transfer of a learned relation pattern to new entity triples; it is not zero-shot discovery of English schema words.\n\nMedian run metrics:\n{med[['exact_mrr','rephrase_mrr','twohop_mrr','inverse_mrr','premise_contains_mrr','premise_maps_mrr','stream_loss','parity_repaired','parity_lost']].to_string()}\n\nChecks:\n{json.dumps(checks,indent=2)}\n\nInterpretation boundary:\nEven a positive two-hop result is not human-like understanding. It is evidence only that a small recurrent learner acquired and retained a portable relational schema whose composed target sentence was omitted for the probe entities. The probe-details CSV records prompt, answer rank, and top-5 tokens for direct inspection.\n'''
    (OUT/f'{tag}_{args.seeds}seed_REPORT.txt').write_text(report); print(report)



## Full natural interference

Run the balanced Shakespeare/Bible test first:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3"])
```

Optional separate runs use `shakespeare` or `bible` for `--corpus`.

In [ ]:
main(["--full", "--corpus", "interleaved", "--seeds", "3"])

## Verified local result

```
Adrianic v4.2 — Direct Semantic / Compositional Probe
THREE-SEED LOCAL VERIFIED PROXY TEST

Purpose
=======
Ask the tiny recurrent learner direct questions after sequential interference. Training remains next-token prediction only. No semantic labels, chain IDs, relation roles, answer keys, or probe targets are supplied to the learner or memory selector.

Controlled relation test
========================
Twenty demonstration chains contain four textual relations: X contains Y; Y maps to Z; X ultimately maps to Z; Y is contained by X.

Ten disjoint probe chains contain ONLY the first two premises. Their composed X->Z sentence and inverse Y->X sentence are omitted. This means a correct two-hop/inverse preference cannot be exact recall of the omitted target sentence.

Median post-interference metrics
================================
                     exact_mrr  rephrase_mrr  twohop_mrr  inverse_mrr  premise_contains_mrr  premise_maps_mrr  stream_loss  parity_repaired  parity_lost
variant                                                                                                                                                 
baseline              0.554407      0.176525    0.030148     0.004834              0.883333          0.565263     0.811936             30.0          0.0
framework             0.602072      0.200195    0.068761     0.004611              1.000000          0.612698     0.802917             30.0          0.0
framework_no_parity   0.638291      0.189327    0.036801     0.005190              1.000000          0.724028     0.786707              0.0         30.0
random_replay         0.664205      0.276764    0.034979     0.004719              1.000000          0.676234     0.776583             33.0          0.0

Two-hop target-specificity control
==================================
For each probe prompt, the correct composed target is compared against a deterministically shuffled wrong target from another chain.

            variant  correct_mrr  shuffled_mrr     delta  correct_rank_better_frac  median_correct_rank  median_shuffled_rank                                                             seed_deltas  positive_seed_deltas
           baseline     0.029235      0.035856 -0.006622                  0.500000                 43.5                  47.0 [-0.0068217559388574975, -0.0033983670406065106, -0.009644858071325011]                     0
      random_replay     0.042679      0.045354 -0.002675                  0.433333                 28.5                  26.5  [-0.0005279038318346321, -0.009342608894024357, 0.0018464980244483897]                     1
          framework     0.065327      0.046735  0.018592                  0.600000                 22.0                  33.0         [0.0341380085837635, 0.013387383302727277, 0.00825061722537277]                     3
framework_no_parity     0.054484      0.058498 -0.004014                  0.466667                 26.0                  31.5      [0.00248534111975926, -0.018769162325201177, 0.004242910794088513]                     2

Transparent chain-memory audit
==============================
The framework retained at least one copy of BOTH premises for 30/30 probe chains across the three seeds.

Checks
======
{
  "framework_exact_above_baseline_median": true,
  "framework_rephrase_above_baseline_median": true,
  "framework_twohop_above_baseline_median": true,
  "framework_twohop_above_random_median": true,
  "framework_twohop_correct_beats_shuffled_all_3_seeds": true,
  "framework_preserves_both_probe_premises_all_chains": true,
  "inverse_transfer_established": false,
  "parity_repaired_faults": true
}

What survived
=============
The framework's median two-hop MRR is 0.068761, versus 0.030148 baseline and 0.034979 random replay. More importantly, the framework preferred the correct composed target over the shuffled wrong target on aggregate, with a positive correct-minus-shuffled MRR delta in all three seeds. Baseline had a negative delta in all three seeds; random replay did not show that all-seed specificity.

What did NOT survive
====================
Inverse-relation transfer is essentially absent: all variants remain near an MRR of roughly 0.005. That probe is a failure, and should remain a failure in the scientific record rather than being folded into a generic 'semantics' claim.

Also, random replay still beats the current selector on median exact-fact and rephrasing retention in this particular proxy test. The interesting advantage is narrower: the framework preserved the paired premises and showed a more target-specific two-hop composition signal.

Interpretation boundary
=======================
This is not human-like understanding. It is the first controlled evidence in this branch that a tiny recurrent model plus the deterministic memory framework can retain a relational schema strongly enough to prefer a composed answer whose target sentence was omitted for the probe entities. The effect is small, only three seeds, and must be repeated under the natural Shakespeare/Bible interference run before being promoted.

```